# RDataFrame basics

In the previous introduction we saw a traditional event-loop analysis:

```python
for event in tree:
    compute something
    apply selections
    fill histograms
```

RDataFrame lets us express the same workflow at a higher level:

```text
TTree -> Define new columns -> Filter events -> Fill histograms / save outputs
```

At the same time, users can just as easily specify custom code that will be executed in the event loop. The framework takes care of the management of the loop over entries as well as low-level details such as I/O and parallelisation.
<img src="data/rdf_1.png" style="width: 75%;">

## Start ROOT and open a dataset

We use a small example ROOT file. It contains a TTree called `Events`.

RDataFrame will read this TTree and allow us to define selections, derived quantities, and histograms.

In [ ]:
import ROOT

treename = "Events"
filename = "data/example_file.root"

df = ROOT.RDataFrame(treename, filename)

print(f"Columns in the dataset: {df.GetColumnNames()}")

## First action: count events

`Count()` counts the number of entries in the dataframe.

`Count()` does not immediately return a Python integer. It returns a *smart pointer for the return type of actions*.
The actual event loop runs when the value is requested, for example with `GetValue()` or when printing the result.

In [ ]:
n_events = df.Count()

print(f"The result handle is: {n_events}")
print(f"The actual number of events is: {n_events.GetValue()}")

## Define a new column

`Define` creates a new column from existing columns.

Here we define

```text
c = a + b
```

This does not modify the original file. It creates a new dataframe node that knows how to compute `c` when needed.

In [ ]:
df_with_c = df.Define("c", "a + b")

print(f"Columns after Define: {df_with_c.GetColumnNames()}")
for var in df_with_c.GetColumnNames():
    print(df_with_c.GetColumnType(var))

## Filter rows

`Filter` keeps only rows that satisfy a condition.

Here we keep only events where `c < 0.5`.

This is the RDataFrame equivalent of writing something like:

```python
if c >= 0.5:
    continue
```

inside a traditional event loop.

In [ ]:
df_selected = df_with_c.Filter("c < 0.5")

n_selected = df_selected.Count()
mean_c = df_selected.Mean("c")

print(f"Selected events: {n_selected.GetValue()}")
print(f"Mean value of c after selection: {mean_c.GetValue():.3f}")

## Display a few rows

`Display` is very useful while developing an analysis. It prints a small table with selected columns.

For large datasets, do not use `Display` to inspect everything. Use it only for a few rows.

In [ ]:
display = df_selected.Display(["a", "b", "c"])
display.Print()

## Fill a histogram

A common analysis output is a histogram.

With an event loop, you would write:

```python
hist.Fill(c)
```

With RDataFrame, you declare what you want:

```python
Histo1D("c")
```

RDataFrame takes care of the event loop.

In [ ]:
#jsroot on
canvas = ROOT.TCanvas()
h_c = df_selected.Histo1D("c")
h_c.Draw()

canvas.Draw()

## Configure the histogram
We can define the histogram name, histogram title, number of bins, axis ranges.

The syntax is:
```python
Histo1D((name, title, nbins, xlow, xup), column)
```

The title string can also contain axis titles:

```text
"title;x-axis;y-axis"
```

In [ ]:
hist_model = (
    "h_c",
    "Distribution of c after selection;c = a + b;Events",
    50,
    -2.0,
    2.0,
)

canvas = ROOT.TCanvas()
h_c_custom = df_selected.Histo1D(hist_model, "c")
h_c_custom.Draw()
canvas.Draw()

## Think about the data flow

The analysis we just wrote can be drawn as a graph:

<img src="data/graph.svg" style="width: 15%;">

This is the main model for RDataFrame.

You do not write the event loop.  
You describe the analysis graph, and RDataFrame executes it.

## Lazy execution

RDataFrame is **lazy**.

This means that transformations such as `Define` and `Filter` do not immediately loop over the data.

The event loop is triggered by **actions**, for example:

- `Count`
- `Mean`
- `Histo1D`

This is important for performance: if you book several actions before requesting their values, RDataFrame can run a single event loop and fill all results at once.

In [ ]:
df_lazy = ROOT.RDataFrame(treename, filename)

# These lines only build the analysis graph.
df_lazy = df_lazy.Define("c", "a + b")
df_lazy = df_lazy.Filter("c < 0.5")

# These lines book actions, but do not necessarily force all computations yet.
count = df_lazy.Count()
mean = df_lazy.Mean("c")
hist = df_lazy.Histo1D(("h_lazy", "Lazy execution example;c;Events", 50, -2, 2), "c")

# Asking for a result triggers the event loop.
print(f"Selected events: {count.GetValue()}")
print(f"Mean c: {mean.GetValue():.3f}")
print(f"The dataset was processed {df_lazy.GetNRuns()} times.")

### Common performance pitfall

Try to **book all actions first**, and only then request the values.

In [ ]:
# Less ideal:
df_bad = ROOT.RDataFrame(treename, filename)
h1 = df_bad.Histo1D("a")
h1.GetValue() # triggers one event loop

h2 = df_bad.Histo1D("b")
h2.GetValue() # triggers another event loop
print(f"The dataset was processed {df_bad.GetNRuns()} times.")

# Better:
df_good = ROOT.RDataFrame(treename, filename)
h1 = df_good.Histo1D("a")
h2 = df_good.Histo1D("b")

h1.GetValue() # one event loop can fill both histograms
h2.GetValue()
print(f"The dataset was processed {df_good.GetNRuns()} times.")

# Operation categories in RDataFrame
There are 3 main types of operations you can perform on RDataFrames:

**Transformations**: manipulate the dataset, return a modified RDataFrame for further processing.

| Transformation    | Description                                                |
|-------------------|------------------------------------------------------------|
| Alias()           | Introduce an alias for a particular column name.           |
| Define()          | Creates  a new column in the dataset.                      |
| Filter()          | Filter rows based on user-defined conditions.              |

**Actions**: aggregate (parts of) the dataset into a result.

| Action                        | Description                                                                          |
|------------------------------------|--------------------------------------------------------------------------------------|
| Count()                            | Return the number of events processed.                                               |
| Display()                          | Provides a printable object representing the dataset contents.                       |
| Graph()                            | Fills a TGraph  with the two columns provided.                                       |
| Histo1D(), Histo2D(), Histo3D()    | Fill a one-, two-, three-dimensional histogram with the processed column values.     |
| Max(), Min()                       | Return the maximum(minimum) of processed column values.                              |
| Snapshot()        | Writes processed data-set to a new TTree.              |
| ...                                | ...  

**Queries**: these methods  query information about your dataset and the RDataFrame status.

| Operation           | Description                                                                              |
|---------------------|------------------------------------------------------------------------------------------|
| GetColumnNames()    | Get the names of all the available columns of the dataset.                               |
| GetColumnType()     | Return the type of a given column as a string.                                           |
| SaveGraph()         | Export the computation graph of an RDataFrame in graphviz format for easy inspection.     |
| ...                 | ...                                                                                      |

### Exercise

Starting from `df`, create a new dataframe that:

1. defines a new column  
   ```text
   d = a - b
   ```
2. keeps only events with  
   ```text
   d > 0
   ```
3. fills a histogram of `d`

Try to write the solution yourself.

In [ ]:
# YOUR SOLUTION